#### Importing libraries

In [1]:
from langchain_chroma import Chroma 
from langchain_openai import OpenAIEmbeddings

import sys 
import os 
sys.path.append(os.path.abspath(".."))
from src.query import (
    get_final_response,
    load_retrievers,
    generate_multi_queries,
    dense_search,
    bm25_search
)
from src.global_settings import CHROMA_PATH, EMBEDDING_FUNCTION

#### Loading ChromaDB

In [2]:
db = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=EMBEDDING_FUNCTION
)

In [3]:
print(f"Number of chunks in the database: {db._collection.count()}")

Number of chunks in the database: 1785


#### Loading Retrievers

In [4]:
chroma_db, bm25, docs = load_retrievers()

#### Generating multiple queries

In [5]:
query = "When can cudaGetDriverEntryPoint return a NULL function pointer even if the API call returns cudaSuccess?"

In [6]:
multi_queries = generate_multi_queries(query)
multi_queries

['Definition and explanation of cudaGetDriverEntryPoint function and its return values.',
 'Typical usage patterns and expected behavior of cudaGetDriverEntryPoint in CUDA applications.',
 'Related CUDA concepts and functions that interact with cudaGetDriverEntryPoint.',
 'Common error scenarios and troubleshooting when cudaGetDriverEntryPoint returns a NULL pointer despite cudaSuccess status.',
 'When can cudaGetDriverEntryPoint return a NULL function pointer even if the API call returns cudaSuccess?']

#### Dense search (OpenAI Embeddings + ChromaDB)

In [7]:
dense_search_docs = dense_search(chroma_db, query)
dense_search_docs

[Document(id='5f1b79f0-e22c-4e61-a91a-51ccfb6d2e29', metadata={'h3': 'Functions Functions', 'h2': '6.31. Driver Entry Point Access', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'chunk_id': 621, 'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html'}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __h

#### Sparse search (BM25)

In [8]:
sparse_search_docs = bm25_search(bm25, docs, query)
sparse_search_docs

[Document(metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'h2': '6.31. Driver Entry Point Access', 'h3': 'Functions Functions', 'chunk_id': 621}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __host__ \u200b cudaError_t cudaGetDriverEntry

#### Query answering

In [9]:
response = get_final_response('When can cudaGetDriverEntryPoint return a NULL function pointer even if the API call returns cudaSuccess?')

In [10]:
response 

{'response': 'The API cudaGetDriverEntryPoint will return cudaSuccess and set the returned funcPtr to NULL if the requested driver function is not supported on the platform or no ABI compatible driver function exists for the requested driver symbol.',
 'urls': ['https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html',
  'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html',
  'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html',
  'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html',
  'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html']}